# **1. Import all modules**

In [ ]:
%matplotlib inline
from modules.WaterLevelHomogenizer import WaterLevelHomogenizer
from modules.wl_q_merger import merge_wl_q
from modules.rating_curve import process_station_rating_curve

# **2. Water Level Homogenization**

## Purpose
- Combines water level data into one consistent series.
- Convert all pressure data to **cmH2O**
- Convert all timestamps to **UTC**
- Correct water pressure using atmospheric pressure:

```python
wl_corr = water_pressure - atmospheric_pressure
```

## Usage

```python
h = WaterLevelHomogenizer(param_file)
cat_df = h.run_station(
    station,
    output_csv=None,
    figure_dir=None,
    plot_single_years=True,
    plot_multi_year=True
)
```

## Input arguments

### `WaterLevelHomogenizer(param_file)`
- `param_file`: YAML parameter file

The user must edit the following fields in `param_file`:
- `main_data_path`: path to the folder containing the input data
- `stations`: station definitions used by the workflow
- for each station:
  - `wl_folder`: relative folder for water-level files
  - `atmo_folder`: relative folder for atmospheric pressure files
  - `base_levels`: observed reference water levels for each year
  - `base_level_uncertainties`: uncertainty values for each base level
  - `base_level_dates`: timestamps of the reference measurements

### `run_station(...)`
- `station`: station name
- `output_csv` *(optional)*: custom output path for the CSV file. Default: `./DISCHARGE/L3/{station}_cat_df.csv`
- `figure_dir` *(optional)*: custom output directory for figures. Default: `./DISCHARGE/figures/`
- `plot_single_years` *(optional, default = `True`)*: save and show single-year plots
- `plot_multi_year` *(optional, default = `True`)*: save and show the concatenated multi-year plot
- `show_plots` *(optional, default = `True`)*: whether to display plots in Jupyter while running the workflow. If `False`, plots are only saved to file and not shown.

Output folders are created automatically if they do not exist.

<div style="border: 2px solid #d9534f; padding: 12px; border-radius: 6px; background-color: #fff3f3;">
<b>Warning</b><br><br>

The program automatically searches for input files in each station's <code>wl_folder</code> and <code>atmo_folder</code>, as defined in <code>param_file</code>. It then uses the internal <code>extract_XX</code> functions to detect and select the correct files.<br><br>

Therefore, input filenames must follow the required naming convention.<br><br>

Filenames must include:
<ul>
  <li>a file type keyword: <code>atmo</code>, <code>water</code>, or <code>cond</code></li>
  <li>a year pattern: <code>YYYY-YYYY</code> (for example <code>2023-2024</code>)</li>
  <li>a processing level: <code>L1</code>, <code>L2</code>, <code>L3</code>, or <code>L4</code></li>
</ul>

Example:
<ul>
  <li><code>./HOBO/data/KG_G354/wl/10662647_water_2023-2024_L2-2.csv</code></li>
  <li><code>./HOBO/data/KG_G354/atmo/21752977_atmo_2023-2024_L2.csv</code></li>
</ul>

If this naming convention is not followed, the workflow may not find the correct input files.
</div>


In [ ]:
h = WaterLevelHomogenizer("00_params_simple_2025-12-11T1003.yml")
h.run_station("KG_G354-new")

## **3. Merge water level and discharge data**

```python
result, result_filtered = merge_wl_q("KG_G354-new")
```

### Input arguments

### `merge_wl_q(...)`
- `station`: station name
- `filename_wl` *(optional)*: custom input path for the concatenated water-level file. Default: `./DISCHARGE/L3/{station}_cat_df.csv`
- `filename_q` *(optional)*: custom input path for the discharge file. Default: `./DISCHARGE/Q-FL/{station}_Q-FL.xlsx`
- `filename_out` *(optional)*: custom output path for the full merged file. Default: `./DISCHARGE/L3/{station}_cat_df_withQv2.csv`
- `filename_out_mini` *(optional)*: custom output path for the filtered WL–Q pairs. Default: `./DISCHARGE/L3/{station}_cat_df_WL-Q.csv`
- `wl_sheet_name` *(optional, default = `"clean"`)*: Excel sheet name used in the discharge file
- `q_time_tolerance` *(optional, default = `"30min"`)*: time window used when selecting discharge measurements around the WL period
- `write_outputs` *(optional, default = `True`)*: whether to write output files

<div style="border: 2px solid #d9534f; padding: 12px; border-radius: 6px; background-color: #fff3f3;">
<b>Warning</b><br><br>

If no custom paths are provided, the program will automatically search for the required input files in <code>./DISCHARGE/L3/</code> and <code>./DISCHARGE/Q-FL/</code>, and save the outputs to <code>./DISCHARGE/L3/</code>.<br><br>

Please make sure that the required input files already exist and that the filenames match the expected station-based naming convention, for example:
<ul>
  <li><code>./DISCHARGE/L3/{station}_cat_df.csv</code></li>
  <li><code>./DISCHARGE/Q-FL/{station}_Q-FL.xlsx</code></li>
</ul>

If the expected files are not found, the workflow will raise an error.
</div>

Output folders are created automatically if they do not exist.

### Output
- `result`: full merged dataframe
- `result_filtered`: dataframe containing only rows where both water level and discharge are available


In [ ]:
merge_wl_q("KG_G354-new")

# **4. Generate rating-curve and discharge**

```python
result = process_station_rating_curve(
    "KG_G354-new",
    H0_min=-10.0,
    H0_max=15.0,
)
```

### Input arguments

### `process_station_rating_curve(...)`
- `station`: station name
- `filename_full` *(optional)*: custom input path for the full WL–Q file. Default: `./DISCHARGE/L3/{station}_cat_df_withQv2.csv`
- `filename_mini` *(optional)*: custom input path for the filtered WL–Q pairs. Default: `./DISCHARGE/L3/{station}_cat_df_WL-Q.csv`
- `output_dir` *(optional)*: custom output directory for rating-curve results. Default: `./DISCHARGE/L4/`
- `figure_dir` *(optional)*: custom output directory for figures. Default: `./DISCHARGE/figures/`
- `H0_min` *(optional)*: lower bound of the H0 search range for this station
- `H0_max` *(optional)*: upper bound of the H0 search range for this station
- `min_points` *(optional, default = `5`)*: minimum number of WL–Q pairs required to fit a rating curve for one year
- `rel_q_uncertainty` *(optional, default = `0.3`)*: relative discharge uncertainty used for the uncertainty envelope
- `min_valid_frac` *(optional, default = `0.5`)*: minimum valid fraction required during H0 fitting
- `save_clean_copy` *(optional, default = `True`)*: whether to save an additional cleaned output file
- `plot_yearly_rating_curves` *(optional, default = `True`)*: save yearly rating-curve plots
- `plot_yearly_timeseries` *(optional, default = `True`)*: save yearly discharge time-series plots
- `plot_all_years_timeseries` *(optional, default = `True`)*: save the combined all-years discharge time-series plot
- `show_plots` *(optional, default = `True`)*: whether to display plots in Jupyter while running the workflow. If `False`, plots are only saved to file and not shown.

<div style="border: 2px solid #d9534f; padding: 12px; border-radius: 6px; background-color: #fff3f3;">
<b>Warning</b><br><br>

If no custom paths are provided, the program will automatically search for the required input files in <code>./DISCHARGE/L3/</code>, and save the output tables to <code>./DISCHARGE/L4/</code> and figures to <code>./DISCHARGE/figures/</code>.<br><br>

Please make sure that the required input files already exist and that the filenames match the expected station-based naming convention, for example:
<ul>
  <li><code>./DISCHARGE/L3/{station}_cat_df_withQv2.csv</code></li>
  <li><code>./DISCHARGE/L3/{station}_cat_df_WL-Q.csv</code></li>
</ul>

If the expected files are not found, the workflow will raise an error.
</div>

Output folders are created automatically if they do not exist.

### Output
- yearly rating-curve output files in `./DISCHARGE/L4/`
- yearly clean copies in `./DISCHARGE/L4/`
- yearly rating-curve plots in `./DISCHARGE/figures/`
- yearly discharge time-series plots in `./DISCHARGE/figures/`
- combined all-years discharge time-series plot in `./DISCHARGE/figures/`

### Return value
- `result`: summary dictionary containing station name, processed years, selected H0 bounds, written files, and output folders


In [ ]:
process_station_rating_curve("KG_G354-new", H0_min=-10.0,H0_max=15.0,)